<a href="https://colab.research.google.com/github/jenitchadha/SDS-510/blob/main/Module_3_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset - phoenix crime stats

For the **basics level project** for mod 3, I will be answering three questions as given in the assignment:
*   Which ZIP codes have the most
reported crimes?
*   What types of crimes are most common in each ZIP code?
*   Which ZIP codes have the largest increases or decreases in reported **crimes** over time?
*   violent vs non-violent crimes.

First, I will download/load the csv file into python. Instead of manually downloading the csv file and then uploading to python, I will directly ask python to load the file for me from kaggle. I am using the python code given on their website directly.



In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mariapushkareva/phoenix-crime-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'phoenix-crime-data' dataset.
Path to dataset files: /kaggle/input/phoenix-crime-data


In [2]:
import os

print(os.listdir(path))

['crimestat (1).csv']


Now, I will save the csv file as a df and look at what my dataset looks like.

In [16]:
import pandas as pd
from io import StringIO

with open(path + "/crimestat (1).csv", "r") as file:
    lines = file.readlines()

df = pd.read_csv(StringIO("".join(lines[:-1])))

df.head()

/tmp/ipykernel_4043/4005859099.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(StringIO("".join(lines[:-1])))


,INC NUMBER,OCCURRED ON,OCCURRED TO,UCR CRIME CATEGORY,100 BLOCK ADDR,ZIP,PREMISE TYPE
0,201600000052855,11/01/2015 00:00,01/09/2016 00:00,MOTOR VEHICLE THEFT,N 43RD AVE & W CACTUS RD,85029.0,SINGLE FAMILY HOUSE
1,201600000594484,11/01/2015 00:00,NaN,RAPE,13XX E ALMERIA RD,85006.0,SINGLE FAMILY HOUSE
2,201500002102327,11/01/2015 00:00,11/01/2015 09:00,LARCENY-THEFT,51XX N 15TH ST,85014.0,APARTMENT
3,201500002101405,11/01/2015 00:00,11/01/2015 05:00,MOTOR VEHICLE THEFT,102XX W MEDLOCK AVE,85307.0,SINGLE FAMILY HOUSE
4,201500002102668,11/01/2015 00:00,11/01/2015 11:50,MOTOR VEHICLE THEFT,69XX W WOOD ST,85043.0,SINGLE FAMILY HOUSE


In [17]:
print(df.shape)
print(df.columns.tolist())

(131676, 7)
['INC NUMBER', 'OCCURRED ON', 'OCCURRED TO', 'UCR CRIME CATEGORY', '100 BLOCK ADDR', 'ZIP', 'PREMISE TYPE']


The above shows that the dataset has 131,676 rows and 7 columns.

question 1:  zip codes with the most reported crime records

In [18]:
zip_counts = df.groupby("ZIP").size().sort_values(ascending=False)

zip_counts.head(10)

,0
ZIP,
85015.0,6963
85008.0,6320
85009.0,5513
85041.0,5477
85051.0,5389
85017.0,5355
85035.0,4752
85043.0,4331
85032.0,4301


It looks like ZIP code 85015 has the most reported crime records (6,963), followed by 85008 (6,320) and 85009 (5,513).

question 2: what types of crimes are most common in each zip code?



In [19]:
crime_by_zip = pd.crosstab(df["ZIP"], df["UCR CRIME CATEGORY"])

crime_by_zip.head()

UCR CRIME CATEGORY,AGGRAVATED ASSAULT,ARSON,BURGLARY,DRUG OFFENSE,LARCENY-THEFT,MOTOR VEHICLE THEFT,MURDER AND NON-NEGLIGENT MANSLAUGHTER,RAPE,ROBBERY
ZIP,,,,,,,,,
85003.0,120,7,179,194,758,128,2,47,78
85004.0,119,4,142,114,1007,126,5,37,96
85006.0,248,20,467,179,1202,291,12,90,152
85007.0,234,21,310,288,758,180,15,64,130
85008.0,516,45,933,515,3196,729,20,104,262


In [20]:
most_common = pd.DataFrame({
    "Most common crime": crime_by_zip.idxmax(axis=1),
    "Count": crime_by_zip.max(axis=1)
})

most_common.head(10)

,Most common crime,Count
ZIP,,
85003.0,LARCENY-THEFT,758
85004.0,LARCENY-THEFT,1007
85006.0,LARCENY-THEFT,1202
85007.0,LARCENY-THEFT,758
85008.0,LARCENY-THEFT,3196
85009.0,LARCENY-THEFT,2184
85012.0,LARCENY-THEFT,463
85013.0,LARCENY-THEFT,1577
85014.0,LARCENY-THEFT,1177


From the above table, we can see that Larceny-theft is the most common crime category in all 10 ZIP codes.

In [35]:
most_common["Most common crime"].value_counts()

,count
Most common crime,
LARCENY-THEFT,69
DRUG OFFENSE,14
AGGRAVATED ASSAULT,7
RAPE,4
MOTOR VEHICLE THEFT,2
BURGLARY,1
ROBBERY,1


Now, I have to check the dates available to compare crime over time.
For this, I will be using the following functions:
- pd.to_datetime() to convert the column into dates.

In [23]:
df["OCCURRED ON"] = pd.to_datetime(df["OCCURRED ON"], errors="coerce")
df["year"] = df["OCCURRED ON"].dt.year
df = df.dropna(subset=["ZIP"])

print(df["year"].value_counts().sort_index())

print(df["OCCURRED ON"].min())
print(df["OCCURRED ON"].max())

year
2015.0    10124
2016.0    63225
2017.0    58008
Name: count, dtype: int64
2015-11-01 00:00:00
2017-11-17 05:50:00


So, in the above code, I get the dates, create a year column, and remove any missing zips. The output shows that the data is from November 1, 2015, to November 17, 2017

I am not sure how to compare across the dates because 2015 and 2017 are less than half years worth of data in the dataset. Maybe I can compare jan-oct 2016 with 2017.

Also, I was struggling with this code and it won't work for some reason. I would see an error each time. So I have to ask AI and use google to help me with it.

In [26]:
comparison = df[
    df["year"].isin([2016, 2017]) &
    (df["OCCURRED ON"].dt.month <= 10)
]

counts = comparison.groupby(["ZIP", "year"]).size().unstack(fill_value=0)

counts.head()

year,2016.0,2017.0
ZIP,,
85003.0,547,687
85004.0,673,693
85006.0,1011,1171
85007.0,808,811
85008.0,2505,2718


So, from the above output we can see that the ZIP code 85003 increased from 547 to 687 records, which is an increase of 140 crime records.
I will now make an additional column to track changes in every ZIP code.

In [27]:
counts["Change"] = counts[2017] - counts[2016]

counts["Percent change"] = (
    counts["Change"] / counts[2016].replace(0, float("nan")) * 100
).round(2)

counts.head()

year,2016.0,2017.0,Change,Percent change
ZIP,,,,
85003.0,547,687,140,25.59
85004.0,673,693,20,2.97
85006.0,1011,1171,160,15.83
85007.0,808,811,3,0.37
85008.0,2505,2718,213,8.50


Finding the largest numberical increases and decreases.

In [30]:
counts.sort_values("Change", ascending=False).head(5)

year,2016.0,2017.0,Change,Percent change
ZIP,,,,
85043.0,1573,1939,366,23.27
85035.0,1741,2065,324,18.61
85009.0,2095,2371,276,13.17
85008.0,2505,2718,213,8.50
85031.0,1079,1286,207,19.18


From the above output, we can see that between January - October 2016 and the same months in 2017, ZIP code 85043 had the largest increase in reported crime records. This is an increase of 366 crime records.

Now, i will find ZIP codes witht he largest decrease in reported crime records.

In [31]:
counts.sort_values("Change").head(5)

year,2016.0,2017.0,Change,Percent change
ZIP,,,,
85015.0,2983,2760,-223,-7.48
85018.0,1179,1010,-169,-14.33
85023.0,1681,1513,-168,-9.99
85016.0,1584,1474,-110,-6.94
85051.0,2222,2138,-84,-3.78


The above output shows that Between January–October 2016 and the same months in 2017, ZIP code 85015 had the largest numerical decrease in reported crime records. This is a decrease of 223 crime records.

Now, I will be grouping non-violent and violent cases together.

In [33]:
violent_crimes = [
    "AGGRAVATED ASSAULT",
    "MURDER AND NON-NEGLIGENT MANSLAUGHTER",
    "RAPE",
    "ROBBERY"
]

df["Crime group"] = df["UCR CRIME CATEGORY"].apply(
    lambda crime: "Violent" if crime in violent_crimes else "Non-violent"
)
df["Crime group"].value_counts()

,count
Crime group,
Non-violent,114458
Violent,17215


There were 114,458 non-violent records and 17,215 violent records.  Non-violent crime records were much more common than violent crime records.

In [37]:
df["PREMISE TYPE"].value_counts().head(10)

,count
PREMISE TYPE,
SINGLE FAMILY HOUSE,25329
APARTMENT,14956
PARKING LOT,14148
STREET / ROADWAY / ALLEY / SIDEWALK,11596
RETAIL BUSINESS,10852
VEHICLE,6844
DRIVEWAY,4609
CONVENIENCE MARKET / STORE,4270
DEPARTMENT / DISCOUNT STORE,3822


From the above output, we can see that Single-family houses had the most reported crimes (25,329), followed by apartments (14,956) and parking lots (14,148).